In [ ]:
import asyncio
import nest_asyncio
from dotenv import load_dotenv
import os

from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI

# Load environment variables from .env (ensure GOOGLE_API_KEY is set)
load_dotenv()

# Apply nest_asyncio to allow running async code in notebooks or interactive environments
nest_asyncio.apply()

# ✅ Define the LLM model (Gemini)
model = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.3)

# ✅ Initialize the client with MCP tools
client = MultiServerMCPClient(
    {

        "weather": {
            "url": "http://localhost:8000/mcp",
            "transport": "streamable_http",
        },
    }
)

# ✅ Define the async workflow
async def main():
    tools = await client.get_tools()

    agent = create_react_agent(
        model,
        tools
    )

    weather_response = await agent.ainvoke(
        {"messages": [{"role": "user", "content": "what is the weather in nyc?"}]}
    )
    print("🌤️ Weather Response:", weather_response)

# ✅ Run it
await main()


🌤️ Weather Response: {'messages': [HumanMessage(content='what is the weather in nyc?', additional_kwargs={}, response_metadata={}, id='ee053f16-8215-435d-b1ce-33fc3a4ce185'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "nyc"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--370127d0-c386-44b8-8fc3-736f16adaf54-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'nyc'}, 'id': 'db1285ae-1898-4eac-b91a-50ea79303662', 'type': 'tool_call'}], usage_metadata={'input_tokens': 19, 'output_tokens': 5, 'total_tokens': 24, 'input_token_details': {'cache_read': 0}}), ToolMessage(content="It's always sunny in New York", name='get_weather', id='cf75c8ab-d741-4fc7-ab64-e61bacea4762', tool_call_id='db1285ae-1898-4eac-b91a-50ea79303662'), AIMessage(content="It's always sunny in New York.", additional_kwarg